# Queen Editor — Colab kurulumu

Bu defterin işi kurmak ve sunmak: Drive'ı bağlar → repoyu klonlar → **ComfyUI'yi kurar** (19 custom
node) → **seçtiğin üreticilerin modellerini indirir** → **Flask** arayüzü servis eder →
**cloudflared** linki basar. Üretim uygulamanın içinde oluyor: bir projeye girip kareler üretirsin,
her kare foto ile başlar ve üstüne video, ses katmanı alabilir; hepsi
`MyDrive/queenEditor/<proje>/` altına düşer.

> **Ne kurulacağını CONFIG'deki üç kutu söyler.** Üçü de kapalı gelir; en az birini işaretle.
> Fotoğraf ~8 GiB · video ~39 GiB · ses ~9 GiB. İşaretlemediğin üretici arayüzdeki **Üreticiler**
> panelinde "kurulu değil" görünür — kurulum bu defterin işidir, panelin değil.

> **Runtime → Change runtime type → T4 GPU** gerekiyor. **Video** seçtiysen T4'ün diski yetmez:
> A100 (Colab Pro) iste.

## Kullanım
1. Bu `queeneditor.ipynb`'yi Colab'a yükle (**File → Upload notebook**).
2. **🔑 Secrets** panelinde `GITHUB_TOKEN` olmalı (fine-grained, yalnız bu repo,
   `Contents: read`). Fotoğraf ya da video kuracaksan `CIVITAI_COOKIE` de gerekiyor (civitai.red →
   giriş yap → F12 → Application → Cookies → `__Secure-civ-token` değeri; ~30 günde bir yenilenir).
   Video üreteceksen üçüncüsü: `XAI_API_KEY` — video prompt'unu yazan dil modeli için; yoksa foto
   üretimi yine çalışır.
3. CONFIG'de kurmak istediğin üreticileri işaretle.
4. **Runtime → Run all** → Drive izni ver → seçimine göre ~10-60 dk → en alttaki linke gir.

In [ ]:
# === CONFIG ===
# Ne kurulacağını buradan seç: Colab bu üç satırı sağdaki formda onay kutusu olarak çizer.
# İşaretlemediğin üreticinin modelleri hiç indirilmez.
#   fotoğraf ~8 GiB · video ~39 GiB (T4'ün diski çoğu zaman yetmez, A100 iste) · ses ~9 GiB
INSTALL_PHOTO = False  #@param {type:"boolean"}
INSTALL_VIDEO = False  #@param {type:"boolean"}
INSTALL_AUDIO = False  #@param {type:"boolean"}

# Asked before anything else: with no producer chosen the app still opens and still serves, but a
# queued job would sit waiting for a producer that never arrives. A second here beats a whole
# setup run.
assert INSTALL_PHOTO or INSTALL_VIDEO or INSTALL_AUDIO, (
    "❌ Hiçbir üretici seçilmedi — yukarıdaki kutulardan en az birini işaretle "
    "(INSTALL_PHOTO / INSTALL_VIDEO / INSTALL_AUDIO) ve hücreyi tekrar çalıştır."
)

# The GitHub token comes from Colab's Secrets store (🔑 in the left sidebar), NOT this cell
# -- set once per Google account, never pasted again, never in the notebook source or git.
# Add a secret named GITHUB_TOKEN (fine-grained, this repo, "Contents: read") and grant this
# notebook access. See README for the token setup.
from google.colab import userdata

try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = ""   # secret missing or access not granted -> the assert below explains the fix

# The notebook and the code it clones are one thing: the clone cell asserts on files this run
# added, so a branch without them stops before anything installs. A dev run points at its own
# branch here and comes back to main when that work lands.
BRANCH       = "main"                       # released work lands in main; a dev run points here
REPO         = "AltanBaysal/Internal-tools" # <owner>/<repo>
CLONE_DIR    = "/content/Internal-tools"    # clone target on Colab's local disk
# The app is started from here, and MMAudio's own weights have to land here too: the library
# resolves ./weights and ./ext_weights against the working directory.
APP_DIR      = f"{CLONE_DIR}/queen-editor"
APP_PORT     = 8000                         # Flask port (matches backend/config.py)
DRIVE_FOLDER = "queenEditor"                # proje kökü (MyDrive altında) — adı buradan değiştir

# === ComfyUI (kurulum + üretim; backend QE_COMFY_URL ile bu adrese konuşur) ===
COMFY_PORT  = 8188
COMFY_ROOT  = "/content/ComfyUI"
COMFY_LOG   = "/content/comfyui.log"
COMFYUI_URL = f"http://127.0.0.1:{COMFY_PORT}"

# Civitai's gated models need the session cookie: two files of the photo group and four of the
# video group come from there. Like GITHUB_TOKEN it comes from Colab Secrets -- this notebook is
# committed, so a pasted session JWT would land in git.
try:
    COOKIE_VALUE = userdata.get("CIVITAI_COOKIE")
except Exception:
    COOKIE_VALUE = ""

# A video's prompt is written by xAI when the job's turn comes; the key comes from Secrets like the
# two above. No assert: a photo-only run needs no language model, and stopping the notebook over a
# key that run never uses would be wrong. Trimmed on the way in -- a value pasted with a trailing
# newline would travel into the Authorization header and come back as a 400.
try:
    XAI_API_KEY = (userdata.get("XAI_API_KEY") or "").strip()
except Exception:
    XAI_API_KEY = ""

assert GITHUB_TOKEN, (
    "❌ GITHUB_TOKEN yok — Colab solundaki 🔑 Secrets panelinden 'GITHUB_TOKEN' adıyla ekle "
    "ve bu notebook'a erişimi aç (fine-grained, yalnız bu repo, Contents: read)."
)
# Only the photo and video groups are gated; a sound-only run sends no cookie at all. Asserted here
# rather than at the download: hearing it now costs a second, hearing it after ComfyUI's install
# costs ten minutes.
if INSTALL_PHOTO or INSTALL_VIDEO:
    assert len(COOKIE_VALUE or "") > 200, (
        "❌ CIVITAI_COOKIE yok/çok kısa — Colab 🔑 Secrets'a 'CIVITAI_COOKIE' adıyla ekle: "
        "civitai.red → giriş → F12 → Application → Cookies → __Secure-civ-token değeri (ES256 JWT)"
    )

# SDXL needs a GPU. A CPU runtime has no driver at all, so nvidia-smi is missing rather than
# failing -- and ComfyUI would come up fine there and then fail on every render.
import subprocess as _sp
try:
    _gpu = _sp.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True)
    _gpu_name = _gpu.stdout.strip() if _gpu.returncode == 0 else ""
except FileNotFoundError:
    _gpu_name = ""
assert _gpu_name, (
    "❌ GPU yok — Runtime → Change runtime type → T4 GPU seç ve Run all'ı yeniden çalıştır"
)

# The model and the address live here now, not only in the app's defaults: the probe below has to
# ask exactly what the app will ask, and two places holding one model name would drift. The app
# reads both from the environment (see the Flask cell); config.py's literals stay as the fallback
# for a local run.
XAI_MODEL = "grok-4.3"
XAI_URL   = "https://api.x.ai/v1/chat/completions"

def xai_probe(key, *, fatal):
    """Ask xAI the smallest question there is, and report what it answers.

    The probe IS the app's own call: xAI documents no endpoint for checking a key, and asking the
    same question the app will ask proves more anyway -- the key, the model name and the reach of
    the service, in one request. So anything but a 2xx means the app's call would fail the same
    way, and the reason printed is xAI's own body, never a guessed one.

    fatal=True stops the run. A video's prompt is written here and there is no manual path, so
    ~39 GiB of video models installed against a broken key is time spent for nothing. A network
    error stops nothing: a timeout says nothing about the key.
    """
    import requests
    try:
        answer = requests.post(
            XAI_URL,
            headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"},
            json={"model": XAI_MODEL,
                  "messages": [{"role": "user", "content": "ping"}],
                  "max_tokens": 1},
            timeout=30,
        )
    except Exception as unreachable:
        print(f"⚠️  xAI yoklanamadı ({type(unreachable).__name__}: {unreachable}) — "
              f"anahtar hakkında bir şey söylenemiyor, koşu sürüyor")
        return
    if answer.status_code // 100 == 2:
        print(f"✓ xAI anahtarı çalışıyor (model: {XAI_MODEL})")
        return
    said = (f"xAI anahtarı reddedildi — HTTP {answer.status_code}, "
            f"xAI yanıtı: {answer.text[:400]}")
    if fatal:
        raise RuntimeError(
            f"❌ {said}\nVideo kuruluyor ama video prompt'unu bu anahtar yazıyor, başka yolu yok. "
            f"console.x.ai'dan yeni anahtar al ve Colab Secrets'taki XAI_API_KEY'i güncelle."
        )
    print(f"⚠️  {said}")
    print("   Video üretmeyeceksen sorun değil — foto üretimi anahtarsız da çalışır.")

_chosen = [name for name, on in (("fotoğraf", INSTALL_PHOTO), ("video", INSTALL_VIDEO),
                                 ("ses", INSTALL_AUDIO)) if on]
print(f"✓ GPU: {_gpu_name}")
print("✓ CONFIG hazır (token Colab Secrets'tan okundu)")
print(f"✓ Kurulacak üretici: {', '.join(_chosen)}")
print(f"✓ Dal: {BRANCH}  |  Repo: {REPO}  |  Hedef: {CLONE_DIR}")
print(f"✓ Proje kökü: MyDrive/{DRIVE_FOLDER}")
# Last of the gates and the only one that costs a request, so everything free runs first.
if not XAI_API_KEY:
    print("✓ xAI anahtarı: yok — video prompt yazılamaz (foto üretimi etkilenmez)")
else:
    xai_probe(XAI_API_KEY, fatal=INSTALL_VIDEO)

In [ ]:
# === Mount Google Drive ===
# Projects ARE Drive folders, so the mount must succeed before the server starts: writing under
# /content/drive without a mount silently lands on Colab's local disk, and those folders die with
# the runtime. The first run opens a Google permission window -- grant it.
import os
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
os.makedirs(DRIVE_ROOT, exist_ok=True)   # first run creates it; later runs reuse it
assert os.path.isdir(DRIVE_ROOT), f"❌ Proje kökü oluşmadı: {DRIVE_ROOT}"
print(f"✓ Drive bağlı — proje kökü: {DRIVE_ROOT}")

In [ ]:
# === Clone (delete-and-reclone: the local tree is disposable, always fetch the latest) ===
# subprocess.run with an argument LIST (not shell=True): the token never reaches the shell
# history or a log line. On failure git's stderr is printed RAW, with the token masked.
import os, shutil, subprocess

def _mask(text):
    """Replace the token with <token> so no output ever carries it."""
    return text.replace(GITHUB_TOKEN, "<token>") if GITHUB_TOKEN else text

if os.path.exists(CLONE_DIR):
    shutil.rmtree(CLONE_DIR)              # no pull/merge -- a fresh clone has one behaviour

clone_url = f"https://{GITHUB_TOKEN}@github.com/{REPO}.git"   # never printed (carries the token)
result = subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--depth", "1", clone_url, CLONE_DIR],
    capture_output=True, text=True,
)
if result.returncode != 0:
    # Raw git output, token masked -- never invent a cause (repo comment rule).
    raise RuntimeError("❌ Klon başarısız:\n" + _mask(result.stderr.strip() or result.stdout.strip()))

# The built frontend ships in the repo (frontend/dist) -- fail loud if it is missing, so a
# forgotten rebuild-and-commit shows up here, not as a blank page.
DIST = os.path.join(CLONE_DIR, "queen-editor", "frontend", "dist", "index.html")
assert os.path.exists(DIST), f"❌ Derlenmiş arayüz yok: {DIST} — frontend'i derleyip commit'le (README)"

# All three graphs ship with the repo (our own copies) -- a forgotten commit shows up here rather
# than at the first render. Two of them are video: the standard one, and the one for a video that
# ends on a chosen picture. Sound has no graph: it runs in the app's own process.
for _name in ("workflow_api.json", "workflow_video_api.json",
              "workflow_video_first_last_api.json"):
    _path = os.path.join(CLONE_DIR, "queen-editor", _name)
    assert os.path.exists(_path), f"❌ Grafik yok: {_path} — {_name} commit'lenmiş mi?"
print("✓ Klon tamam (derlenmiş arayüz + üç grafik mevcut)")

In [ ]:
# === Shared helpers — log + fail-loud run + model validation ===
# Used by the custom node cell and the model download cell below; defined once (DRY).
import os, json, time, struct, subprocess

# Neither cell below needs CONFIG for its paths, so without this gate a failed CONFIG cell stays
# invisible until the app starts -- after a ~10 min install.
assert "COMFY_ROOT" in globals(), "❌ Önce 1) CONFIG hücresini çalıştır"

def log(msg, level="INFO"):
    icons = {"INFO": "ℹ️ ", "OK": "✅", "WARN": "⚠️ ", "ERR": "❌"}
    print(f"{icons.get(level, '·')} [{time.strftime('%H:%M:%S')}] {msg}")

def human(b):
    """Bytes -> human-readable size (e.g. 1.5GB)."""
    for u in ["B", "KB", "MB", "GB"]:
        if b < 1024:
            return f"{b:.1f}{u}"
        b /= 1024
    return f"{b:.1f}TB"

def head_text(path, limit=4000):
    """First bytes of a file as raw text — the response body, printed as-is, not interpreted."""
    if not os.path.exists(path):
        return "(dosya yok)"
    size = os.path.getsize(path)
    with open(path, "rb") as f:
        text = f.read(limit).decode("utf-8", errors="replace")
    return text + (f"\n… (+{human(size - limit)})" if size > limit else "")

def run(cmd, label, cwd=None, timeout=3600):
    """Run a command; non-zero exit or timeout -> RuntimeError with the command's own stderr.

    The single gate for download failures: the downloader exits non-zero on an HTTP error, on a
    transfer that ends before the announced length, and on a full disk.
    """
    try:
        r = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd,
                           capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        raise RuntimeError(f"{label}: timeout ({timeout}s)")
    if r.returncode != 0:
        tail = "\n".join((r.stderr or r.stdout or "").strip().splitlines()[-5:])
        raise RuntimeError(f"{label}: exit {r.returncode}\n{tail}")
    return r.stdout

def check_safetensors(path):
    """State of a model file -> ("ok" | "partial" | "invalid", msg).

    The expected total size is computed from the file itself: a safetensors file is
    [8-byte LE header length][header JSON][tensor data], and the header's data_offsets say where
    the tensor data ends. No Content-Length, no HEAD request (HF's Xet CDN answers HEAD with 403
    while serving the GET fine, so a HEAD-based size check reads the error body as the size).

    ok      -> header parses and the file is exactly as long as its header says
    partial -> valid prefix, shorter than expected: safe to resume
    invalid -> empty / error page / longer than expected: garbage, stop
    """
    if not os.path.exists(path):
        return "invalid", "missing"
    size = os.path.getsize(path)
    if size < 8:
        return "invalid", f"too small ({human(size)})"

    with open(path, "rb") as f:
        header_len = struct.unpack("<Q", f.read(8))[0]
        if not (0 < header_len < 200_000_000):
            return "invalid", f"bad header length ({header_len})"
        if 8 + header_len > size:
            return "partial", f"header incomplete ({human(size)})"
        try:
            header = json.loads(f.read(header_len).decode("utf-8"))
        except (UnicodeDecodeError, json.JSONDecodeError) as e:
            return "invalid", f"header parse failed ({type(e).__name__}, {human(size)})"

    ends = [v["data_offsets"][1] for k, v in header.items()
            if k != "__metadata__" and isinstance(v, dict) and "data_offsets" in v]
    if not ends:                        # metadata-only header: nothing to measure against
        return "ok", f"{human(size)}, no tensor offsets"

    expected = 8 + header_len + max(ends)
    if size == expected:
        return "ok", f"{human(size)}, {len(ends)} tensors"
    if size < expected:
        return "partial", f"{size:,} / {expected:,} bytes"
    return "invalid", f"too long: {size:,} / {expected:,} bytes"

print("✓ Ortak yardımcılar hazır (log, run, human, head_text, check_safetensors)")

## ComfyUI + Custom Node'lar (20)

İki grafiğin ihtiyacı olan paketler + Manager: ilk dokuzu foto grafiği, kalanı video grafiği için.
Liste grafiklerin node künyelerinden çıkarıldı; kalan node'lar comfy-core, kurulum istemez. Biri
başarısız olursa hücre `RuntimeError` ile durur (fail-loud).

> **Node'lar seçime bağlı değil, hepsi kurulur.** Ağır olan modeller; node'lar birkaç dakika. Ses
> ise burada hiç yok: ComfyUI'de üretilmiyor, motoru aşağıda kendi hücresinde kuruluyor.

In [ ]:
%cd /content

# === System deps + ComfyUI ===
# ffmpeg is the app's own tool, not ComfyUI's: the export joins the videos with it and a sound job
# cuts the video it reads with it.
!apt-get install -y aria2 ffmpeg > /dev/null 2>&1
![ -d ComfyUI ] || git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!git pull -q
!pip install -q -r requirements.txt
!pip install -q opencv-python imageio imageio-ffmpeg

# === Custom nodes (fail-loud: clone or pip failure -> RuntimeError) ===
import os
%cd /content/ComfyUI/custom_nodes

# (folder, repo) — trailing comment = what the node provides to a graph this app ships.
# Two graphs, one ComfyUI: the photo graph needs the first block, the video graph the second.
CUSTOM_NODES = [
    ("ComfyUI-Manager",           "https://github.com/ltdrdata/ComfyUI-Manager.git"),          # detect missing nodes in the UI
    ("rgthree-comfy",             "https://github.com/rgthree/rgthree-comfy.git"),             # Power Lora Loader, Seed, Fast Groups Bypasser, Image Comparer
    ("ComfyUI-Impact-Pack",       "https://github.com/ltdrdata/ComfyUI-Impact-Pack.git"),      # FaceDetailer, wildcard prompts, SAMLoader, switches
    ("ComfyUI-Impact-Subpack",    "https://github.com/ltdrdata/ComfyUI-Impact-Subpack.git"),   # UltralyticsDetectorProvider
    ("ComfyUI-Easy-Use",          "https://github.com/yolain/ComfyUI-Easy-Use.git"),           # easy int/float, easy hiresFix, easy cleanGpuUsed
    ("ComfyUI-Custom-Scripts",    "https://github.com/pythongosssss/ComfyUI-Custom-Scripts.git"),  # MathExpression
    ("ComfyUI_UltimateSDUpscale", "https://github.com/ssitu/ComfyUI_UltimateSDUpscale.git"),   # UltimateSDUpscale (tiled Remacri upscale)
    ("ComfyUI-KJNodes",           "https://github.com/kijai/ComfyUI-KJNodes.git"),             # ImageResizeKJv2, ColorMatch
    ("ComfyUI-ppm",               "https://github.com/pamparamm/ComfyUI-ppm.git"),             # CLIPTextEncodeBREAK -- the positive path splits on BREAK instead of encoding it as a word
    # --- the video graph (workflow_video_api.json) ---
    ("comfy_mtb",                 "https://github.com/melMass/comfy_mtb.git"),                 # Pick From Batch, RIFEInterpolation
    ("ComfyUI-VideoHelperSuite",  "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git"),  # VHS_VideoCombine (the mp4 comes out here)
    ("ComfyUI-WanVideoWrapper",   "https://github.com/kijai/ComfyUI-WanVideoWrapper.git"),     # Wan video nodes
    ("ComfyUI-GGUF",              "https://github.com/city96/ComfyUI-GGUF.git"),               # UnetLoaderGGUF -- unreachable in the graph, but ComfyUI validates every node it is sent
    ("ComfyMath",                 "https://github.com/evanspearman/ComfyMath.git"),            # ComfyMathExpression (seconds -> frames)
    ("ComfyUI-Frame-Interpolation", "https://github.com/Fannovel16/ComfyUI-Frame-Interpolation.git"),  # RIFE
    ("ComfyUI-VFI",               "https://github.com/GACLove/ComfyUI-VFI.git"),               # frame interpolation
    ("ComfyUI_Comfyroll_CustomNodes", "https://github.com/Suzie1/ComfyUI_Comfyroll_CustomNodes.git"),  # CR Float To Integer
    ("ComfyUI-mxToolkit",         "https://github.com/Smirnov75/ComfyUI-mxToolkit.git"),       # mxSlider2D (resolution)
    ("ComfyUI-NAG",               "https://github.com/scottmudge/ComfyUI-NAG.git"),            # KSamplerWithNAG (Advanced)
    ("comfyui-adaptiveprompts",   "https://github.com/Alectriciti/comfyui-adaptiveprompts.git"),  # PromptGenerator (the video prompt lands here)
]

for name, url in CUSTOM_NODES:
    if os.path.exists(name) and os.listdir(name):
        log(f"{name}: zaten var")
        continue
    log(f"{name}: cloning...")
    # --recurse-submodules: UltimateSDUpscale vendors its upstream repo as a git submodule;
    # an empty submodule folder makes the node import fail. Harmless for the others.
    run(["git", "clone", "--depth", "1", "--recurse-submodules", url, name], f"clone {name}", timeout=180)
    if not os.listdir(name):                 # clone reported success but folder is empty
        raise RuntimeError(f"{name}: klon sonrası klasör boş")
    req = f"/content/ComfyUI/custom_nodes/{name}/requirements.txt"
    if os.path.exists(req):
        run(f"pip install -q -r {req}", f"pip install {name}", timeout=300)  # install node deps

log(f"{len(CUSTOM_NODES)} custom node hazır", "OK")

## Modeller — seçilen üreticiler, önce gated probe (~8 / ~39 / ~9 GiB)

CONFIG'de işaretlediğin grupların dosyaları iner; işaretlemediğin hiç denenmez. İndirmeden önce
**disk ölçülür**: yer yetmiyorsa hücre gerçek sayılarla durur, yarım dosya bırakmaz.

Gated erişim **ağır indirmeden önce** doğrulanır (ilk 1 KB): cookie ölmüşse GiB'larca dosyaya
başlamadan, Civitai'nin **gerçek yanıtıyla** durur. Bozuk/eksik dosyada hücre durur; bozuk dosya
silinmez, inceleme için diskte kalır. Dosyalar grafiğin beklediği adlarla iner — ad tutmazsa render
"model bulunamadı" ile düşer.

> **Model eklemek:** ilgili gruba (`CIVITAI_PHOTO`, `OPEN_PHOTO`, `CIVITAI_VIDEO`, `OPEN_VIDEO`,
> `OPEN_AUDIO`) bir satır ekle, yeter — checkpoint klasörüne inen her `.safetensors` arayüzdeki
> **Model** listesinde kendiliğinden görünür. Uygulama hangi modellerin kurulu olduğunu bilmez,
> ComfyUI'ye sorar. Ama arayüzün **Üreticiler** paneli grubun tamamını sayıyor: yeni dosya bir
> üreticinin çalışması için gerekiyorsa `backend/features/producers/domain/model_groups.py`'ye de
> eklenmeli.

In [ ]:
import os, glob, shutil

# === Target folders ===
COMFY = COMFY_ROOT
CKPT = f"{COMFY}/models/checkpoints"
LORA = f"{COMFY}/models/loras"
UPSC = f"{COMFY}/models/upscale_models"
BBOX = f"{COMFY}/models/ultralytics/bbox"   # UltralyticsDetectorProvider lists files as "bbox/<name>"
SAMS = f"{COMFY}/models/sams"
DIFF = f"{COMFY}/models/diffusion_models"
VAE  = f"{COMFY}/models/vae"
TENC = f"{COMFY}/models/text_encoders"
CLIPV = f"{COMFY}/models/clip_vision"       # only the first-last video graph loads from here
# ComfyUI never reads this one: sound runs in the app's own process. It lives under the same root
# because the panel and the sampler both hang off it, and a second root for a single file would be
# the same knowledge written twice.
MMAU = f"{COMFY}/models/mmaudio"
for d in [CKPT, LORA, UPSC, BBOX, SAMS, DIFF, VAE, TENC, CLIPV, MMAU]:
    os.makedirs(d, exist_ok=True)

def check_binary(path, min_bytes):
    """State of a .pt/.pth model -> ("ok" | "partial" | "invalid", msg).

    Torch pickle/zip files carry no self-describing total length (unlike safetensors), so the
    honest cheap checks are: the head is not an HTML/JSON error page, and the size clears a loose
    floor (guards against truncated/error downloads, not an exact size). Below the floor counts
    as "partial" so an interrupted download stays resumable.
    """
    if not os.path.exists(path):
        return "invalid", "missing"
    size = os.path.getsize(path)
    with open(path, "rb") as f:
        head = f.read(16)
    if head[:1] in (b"<", b"{"):
        return "invalid", f"error page? ({human(size)})"
    if size < min_bytes:
        return "partial", f"{human(size)} < taban {human(min_bytes)}"
    return "ok", human(size)

# === Single download function — shared flow for HF (aria2c) and Civitai (curl) (DRY) ===
def fetch(url, target_dir, filename, label, *, parallel, headers=None, validate=None):
    """Download + validate a model; anything invalid stops the run (fail-loud, nothing deleted).

    parallel=True -> aria2c (fast for large HF files), False -> curl (Civitai, login cookie).
    validate -> (path) -> (state, msg); default check_safetensors. .pt/.pth files pass a
    check_binary lambda because they have no self-describing length.
    Downloads land in <target>.part and are renamed only once the validator says "ok", so
    ComfyUI never sees a half-written file under the real model name.

    On failure the raw HTTP exchange is printed, not a summary of it: curl runs with
    --fail-with-body (non-zero exit, but the response body is kept instead of discarded) and -D
    (every response header of the redirect chain), so a Civitai 401/403 shows the server's own
    headers and body verbatim.
    """
    validator = validate or check_safetensors
    target = os.path.join(target_dir, filename)
    part = target + ".part"
    hdrs = f"/tmp/{filename}.headers"

    if os.path.exists(target):
        state, msg = validator(target)
        if state == "ok":
            log(f"{label}: zaten var ({msg})")
            return
        raise RuntimeError(f"{label}: {state} — {msg}\n{target}\n--- file head ---\n{head_text(target)}")

    resume = False
    if os.path.exists(part):
        state, msg = validator(part)
        if state == "invalid":
            # Resuming onto garbage would append good bytes to it and hide the problem.
            raise RuntimeError(f"{label}: .part {state} — {msg}\n{part}\n--- file head ---\n{head_text(part)}")
        if state == "ok":
            log(f"{label}: .part zaten tam ({msg}) — indirilmiyor")
        else:
            log(f"{label}: .part'tan devam ({msg})")
            resume = True

    if not os.path.exists(part) or resume:
        log(f"{label}: iniyor")
        if parallel:
            cmd = ["aria2c", "-x", "16", "-s", "16", "-k", "1M", "--continue=true",
                   "--console-log-level=warn", "--auto-file-renaming=false",
                   "--allow-overwrite=true", "-d", target_dir, "-o", os.path.basename(part)]
            if headers:
                cmd += ["--header", headers]
        else:
            cmd = ["curl", "-L", "-C", "-", "--fail-with-body", "--max-time", "1800",
                   "-D", hdrs, "-o", part]
            if headers:
                cmd += ["-H", headers]
        cmd.append(url)
        try:
            run(cmd, label, timeout=3600)
        except RuntimeError as e:
            raise RuntimeError(
                f"{e}\n{url.split('?')[0]}\n"
                f"--- response headers ---\n{head_text(hdrs)}\n"
                f"--- response body ---\n{head_text(part)}"
            ) from None

    state, msg = validator(part)
    if state != "ok":
        raise RuntimeError(f"{label}: {state} — {msg}\n{part}\n{url.split('?')[0]}\n"
                           f"--- response headers ---\n{head_text(hdrs)}\n"
                           f"--- file head ---\n{head_text(part)}")
    os.replace(part, target)
    log(f"{label}: indirildi ve doğrulandı ({msg})", "OK")

# Civitai auth: session cookie ONLY. A ?token= API key authenticates the request as that key's
# account -> creator-gated assets answer 401.
# Host = civitai.RED: the cookie is same-origin there. Sending it to .com is cross-domain and
# returns the login+turnstile page instead of the file.
def civitai_url(version_id):
    return f"https://civitai.red/api/download/models/{version_id}"

def cookie_header():
    return f"Cookie: __Secure-civ-token={COOKIE_VALUE}"

def civitai_probe(version_id, label):
    """Fail-fast: range-download the first 1KB to verify gated access BEFORE the heavy download.
    On non-2xx or a login wall, surface Civitai's ACTUAL response body -- no hardcoded guesses.
    """
    out = "/content/_probe.bin"
    code = (run(["curl", "-sL", "--max-time", "60", "-r", "0-1023",
                 "-H", cookie_header(), "-w", "%{http_code}", "-o", out,
                 civitai_url(version_id)], f"probe {label}") or "").strip()[-3:]
    body = b""
    if os.path.exists(out):
        with open(out, "rb") as f:
            body = f.read(512)
        os.remove(out)
    # success = 2xx AND the body is real binary (safetensors), not an HTML/JSON error page
    if code.startswith("2") and not body.startswith(b"<") and not body.startswith(b'{"'):
        log(f"{label}: erişim OK", "OK")
        return
    raise RuntimeError(f"❌ {label}: HTTP {code} — Civitai yanıtı: "
                       f"{body.decode('utf-8', 'replace').strip() or '(boş gövde — binary değil)'}")

# === What each producer needs. Data only: which list runs is the checkboxes' decision, below. ===
# Civitai rows: (version_id, target_dir, filename, label)
# Open rows:    (url, target_dir, filename, label, min_bytes) -- min_bytes None means safetensors,
#               a number means .pt/.pth, which carries no length of its own (check_binary).
# Every filename is the name the graph loads by, which is not always the name at the source.

CIVITAI_PHOTO = [
    (2744564, CKPT, "nova3DCGXL_ilV90.safetensors",               "Nova 3DCG XL IL v9.0"),
    (1552087, LORA, "USNR_STYLE_ILL_V1_lokr3-000024.safetensors", "USNR STYLE ILL v1.0"),
]
# The photo graph's default-ON FaceDetailer branch loads the detector + SAM at startup; the
# bypassed Ultimate SD Upscale branch reads Remacri the moment the user enables it.
OPEN_PHOTO = [
    ("https://huggingface.co/FacehugmanIII/4x_foolhardy_Remacri/resolve/main/4x_foolhardy_Remacri.pth",
     UPSC, "4x_foolhardy_Remacri.pth", "Remacri 4x upscaler", 50_000_000),
    ("https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov9c.pt",
     BBOX, "face_yolov9c.pt", "Yuz dedektoru (yolov9c)", 40_000_000),
    ("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth",
     SAMS, "sam_vit_b_01ec64.pth", "SAM ViT-B", 300_000_000),
]

WAN22 = "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files"
WAN21 = "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files"
# The video graph has both distill LoRAs switched on in its Power Lora Loader -- I2V v2.0 does not
# ship lightx2v merged, so they have to be on disk. VAELoader asks for 'Wan2_1_VAE_fp32', which is
# not what the source calls the file: ComfyUI finds a model by its name on disk.
OPEN_VIDEO = [
    (f"{WAN22}/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors",
     LORA, "wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors", "Lightx2v I2V HIGH", None),
    (f"{WAN22}/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors",
     LORA, "wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors", "Lightx2v I2V LOW", None),
    (f"{WAN21}/vae/wan_2.1_vae.safetensors",
     VAE, "Wan2_1_VAE_fp32.safetensors", "Wan2.1 VAE", None),
    (f"{WAN21}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
     TENC, "umt5_xxl_fp8_e4m3fn_scaled.safetensors", "UMT5-XXL", None),
    # Read by the first-last graph alone, through its CLIPVisionLoader. Still part of the video
    # group: one producer, and a producer is installed or it is not.
    (f"{WAN21}/clip_vision/clip_vision_h.safetensors",
     CLIPV, "clip_vision_h.safetensors", "CLIP Vision H", None),
]
# UNETLoader asks for the checkpoint pair, the Power Lora Loader for the Animations pair; Civitai
# serves all four under its own file names, so each lands under the name the graph names.
CIVITAI_VIDEO = [
    (2513182, DIFF, "SmoothMix_I2V_v2_High.safetensors",         "SmoothMix I2V v2 HIGH"),
    (2513186, DIFF, "SmoothMix_I2V_v2_Low.safetensors",          "SmoothMix I2V v2 LOW"),
    (2376136, LORA, "SmoothMix_Animations_XXX_High.safetensors", "SmoothMix Animations XXX HIGH"),
    (2376143, LORA, "SmoothMix_Animations_XXX_Low.safetensors",  "SmoothMix Animations XXX LOW"),
]

# The sound fine-tune, and only that: MMAudio's own vae, synchformer and base checkpoint come down
# with the library in the cell below, which is what knows where it keeps them.
OPEN_AUDIO = [
    ("https://huggingface.co/phazei/NSFW_MMaudio/resolve/main/"
     "mmaudio_large_44k_nsfw_gold_8.5k_final_fp16.safetensors",
     MMAU, "mmaudio_large_44k_nsfw_gold_8.5k_final_fp16.safetensors", "MMAudio NSFW fine-tune",
     None),
]

# === What this run installs (the checkboxes, and nothing else) ===
civitai_jobs = (CIVITAI_PHOTO if INSTALL_PHOTO else []) + (CIVITAI_VIDEO if INSTALL_VIDEO else [])
open_jobs = ((OPEN_PHOTO if INSTALL_PHOTO else [])
             + (OPEN_VIDEO if INSTALL_VIDEO else [])
             + (OPEN_AUDIO if INSTALL_AUDIO else []))

# Rounded up on purpose: an estimate that is too low fills the disk and leaves half-written files,
# while one that is too high costs a warning. Sound counts the library's own ~7 GiB too, which
# lands in the cell below rather than here.
SIZES = [(INSTALL_PHOTO, 8, "fotoğraf"), (INSTALL_VIDEO, 39, "video"), (INSTALL_AUDIO, 9, "ses")]
HEADROOM_GIB = 5    # renders, exports and .part files share this disk
need = sum(gib for on, gib, _ in SIZES if on)
free = shutil.disk_usage("/content").free / 1024**3
log(f"Seçim: {', '.join(name for on, _, name in SIZES if on)} — ~{need} GiB "
    f"| Diskte boş: {free:.1f} GiB")
if free < need + HEADROOM_GIB:
    raise RuntimeError(
        f"❌ Disk yetmiyor: ~{need} GiB model + {HEADROOM_GIB} GiB pay gerekiyor, "
        f"{free:.1f} GiB boş. Daha az üretici seç ya da diski daha büyük bir runtime aç "
        f"(video tek başına ~39 GiB)."
    )

# 1) Fail-fast: verify gated access before spending the download time
if civitai_jobs:
    log(f"Gated probe: {len(civitai_jobs)} asset")
    for vid, d, fn, label in civitai_jobs:
        civitai_probe(vid, label)

# 2) Open downloads (aria2c)
for url, d, fn, label, floor in open_jobs:
    fetch(url, d, fn, label, parallel=True,
          validate=(lambda p, m=floor: check_binary(p, m)) if floor else None)

# 3) Civitai — parallel=False: aria2c forwards the cookie to the B2 store on redirect and gets 403,
#    curl drops it cross-host and gets through.
for vid, d, fn, label in civitai_jobs:
    fetch(civitai_url(vid), d, fn, label, parallel=False, headers=cookie_header())

# === Summary (reaching here means everything downloaded + validated) ===
folders = []
if INSTALL_PHOTO:
    folders += [("checkpoints", CKPT, "*.safetensors"), ("upscale_models", UPSC, "*.pth"),
                ("ultralytics/bbox", BBOX, "*.pt"), ("sams", SAMS, "*.pth")]
if INSTALL_VIDEO:
    folders += [("diffusion_models", DIFF, "*.safetensors"), ("vae", VAE, "*.safetensors"),
                ("text_encoders", TENC, "*.safetensors"), ("clip_vision", CLIPV, "*.safetensors")]
if INSTALL_PHOTO or INSTALL_VIDEO:      # both groups write loras
    folders += [("loras", LORA, "*.safetensors")]
if INSTALL_AUDIO:
    folders += [("mmaudio", MMAU, "*.safetensors")]
for title, folder, pattern in folders:
    print(f"\n📂 {title}/")
    for f in sorted(glob.glob(f"{folder}/{pattern}")):
        print(f"   {human(os.path.getsize(f))}  {os.path.basename(f)}")
log("Seçilen modeller indirildi ve doğrulandı", "OK")

## Ses motoru (yalnız `INSTALL_AUDIO` işaretliyse)

Ses ComfyUI'de üretilmiyor: **MMAudio uygulamanın kendi sürecinde** çalışıyor, yani `import mmaudio`
orada çalışmak zorunda. Bu yüzden kutu işaretliyse kütüphane klonlanıp kuruluyor.

Ardından MMAudio'nun **kendi ağırlıkları** (vae, synchformer, taban checkpoint — ~7 GiB) iniyor.
Kütüphane bunları zaten kendi indiriyor; buradaki iş sırayı öne almak, yoksa ilk ses işi kuyrukta
sessizce bu indirmeyi bekletirdi. Dosyalar uygulamanın çalıştığı klasöre iner — MMAudio bu yolları
çalışma dizinine göre çözüyor, başka yere inen dosya uygulama için yok sayılır.

In [ ]:
# === Ses motoru — MMAudio kütüphanesi ===
# Sound is the one producer that is not a ComfyUI graph. Installed here, before the app starts: a
# library that appears after a process has begun is not visible to it.
import os

MMAUDIO_DIR = "/content/MMAudio"

if not INSTALL_AUDIO:
    log("Ses motoru: atlandı (INSTALL_AUDIO kapalı)")
else:
    if not os.path.isdir(MMAUDIO_DIR):
        run(["git", "clone", "--depth", "1", "https://github.com/hkchengrex/MMAudio.git",
             MMAUDIO_DIR], "clone MMAudio", timeout=300)
    # Editable install: the clone stays the package's source, so nothing is copied twice.
    run(["pip", "install", "-e", ".", "-q"], "pip install MMAudio", cwd=MMAUDIO_DIR, timeout=1800)
    log("MMAudio kütüphanesi kuruldu", "OK")

In [ ]:
# === Ses motoru — MMAudio'nun kendi ağırlıkları (~7 GiB) ===
# A cell of its own, not the one above: a package installed by pip in a cell may not be importable
# yet inside that same cell.
import os, sys

if not INSTALL_AUDIO:
    log("MMAudio ağırlıkları: atlandı (INSTALL_AUDIO kapalı)")
else:
    assert os.path.isdir(APP_DIR), f"❌ Uygulama klasörü yok: {APP_DIR} — önce klon hücresini çalıştır"
    # `pip install -e .` registers MMAudio through a .pth file in site-packages, and .pth files are
    # read when a Python process STARTS -- this kernel started long before the install, so it never
    # saw it. Pointing at the clone is what makes the import work in this already-running kernel.
    # The app does not need this: it is a new process and reads the .pth on its own startup.
    if MMAUDIO_DIR not in sys.path:
        sys.path.insert(0, MMAUDIO_DIR)
    # MMAudio resolves ./weights and ./ext_weights against the WORKING DIRECTORY, and the app is
    # started from APP_DIR -- so this is the only folder where a download counts as installed.
    # Anywhere else and the app would fetch the same files again on its first sound job.
    _cwd = os.getcwd()
    os.chdir(APP_DIR)
    try:
        from mmaudio.eval_utils import all_model_cfg
        all_model_cfg["large_44k"].download_if_needed()   # the app's own model (mmaudio_sampler.py)
    finally:
        os.chdir(_cwd)
    log(f"MMAudio ağırlıkları hazır → {APP_DIR}", "OK")

## ComfyUI'yi başlat (arka planda)

ComfyUI subprocess olarak kalkar; arayüzün backend'i `QE_COMFY_URL` ile bu adrese konuşur. **90 sn
içinde hazır olmazsa** hücre log'un son 30 satırını basıp durur — sonraki hücre ölü sunucuya
çalışmasın. Tünel yok: ComfyUI'ın kendi arayüzü açılmıyor.

In [ ]:
import subprocess, time, os, urllib.request

# Re-run safety: kill the previous instance before starting a new one
os.system("pkill -f 'python main.py' 2>/dev/null")
time.sleep(2)

# === Start in background (logs to file) ===
# No --enable-manager: nothing opens the UI here, and a missing node already failed loudly during
# install. No tunnel either -- the render cell talks to localhost.
comfy_log = open(COMFY_LOG, "w")
proc = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT)],
    cwd=COMFY_ROOT, stdout=comfy_log, stderr=subprocess.STDOUT,
)
log(f"ComfyUI başlatıldı (PID {proc.pid}), log: {COMFY_LOG}")

# === Ready? max 90s — otherwise fail-loud with the server's own log ===
for i in range(45):
    time.sleep(2)
    try:
        urllib.request.urlopen(f"{COMFYUI_URL}/system_stats", timeout=2)
        log(f"ComfyUI hazır ({(i + 1) * 2}s)", "OK")
        break
    except Exception:
        pass
else:
    with open(COMFY_LOG) as f:
        print("".join(f.readlines()[-30:]))
    raise RuntimeError("❌ ComfyUI 90 sn içinde başlamadı — yukarıdaki log'a bak")

In [ ]:
# === Start Flask (background) + cloudflared tunnel ===
# Flask serves the pre-built frontend/dist and /api. It runs as a module from queen-editor/ so
# `backend` resolves as a package. The cell stays OPEN (tail -f): if it ends, Colab calls the
# runtime idle and kills the tunnel. No npm/build here -- the UI ships built (ComfyUI pattern).
import subprocess, time, os, re, urllib.request

FLASK_LOG = "/content/flask.log"      # APP_DIR CONFIG'de: ses ağırlıkları da oraya iniyor

# Re-run safety: kill previous instances before starting new ones
subprocess.run(["pkill", "-f", "backend.main"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(2)

logf = open(FLASK_LOG, "w")
# The backend reads its Drive root, its ComfyUI address, the model tree the cells above installed
# into, and its xAI settings from the environment (backend/config.py) -- all decided in the cells
# above, not hardcoded in the app. The Civitai cookie is not among them: the app downloads nothing.
# The model and the address travel with the key, so the probe in CONFIG and the app ask the same
# service the same question.
flask_env = {**os.environ, "QE_DRIVE_ROOT": DRIVE_ROOT, "QE_COMFY_URL": COMFYUI_URL,
             "QE_COMFY_ROOT": COMFY_ROOT, "QE_XAI_API_KEY": XAI_API_KEY or "",
             "QE_XAI_MODEL": XAI_MODEL, "QE_XAI_URL": XAI_URL}
subprocess.Popen(["python", "-m", "backend.main"], cwd=APP_DIR, env=flask_env,
                 stdout=logf, stderr=subprocess.STDOUT)

ok = False
for i in range(45):
    time.sleep(2)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{APP_PORT}/api/health", timeout=2)
        ok = True
        break
    except Exception:
        pass
if not ok:
    print("".join(open(FLASK_LOG).readlines()[-30:]))
    raise RuntimeError("❌ Flask 90 sn içinde /api/health'e cevap vermedi — yukarıdaki log'a bak")
print(f"✓ Flask ayakta ({(i + 1) * 2}s)")

if not os.path.isfile("/content/cloudflared"):
    subprocess.run(["wget", "-q", "-O", "/content/cloudflared",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
    subprocess.run(["chmod", "+x", "/content/cloudflared"], check=True)

tunlog = "/content/cloudflared.log"
# --protocol http2: Colab throttles the default QUIC. Same photo, 17.74 s -> 0.18 s.
subprocess.Popen(["/content/cloudflared", "tunnel", "--protocol", "http2",
                  "--url", f"http://127.0.0.1:{APP_PORT}"],
                 stdout=open(tunlog, "w"), stderr=subprocess.STDOUT)
link = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists(tunlog):
        m = re.search(r"https://[-\w.]+trycloudflare\.com", open(tunlog).read())
        if m:
            link = m.group(0)
            break
if not link:
    print(open(tunlog).read()[-1000:] if os.path.exists(tunlog) else "(cloudflared log yok)")
    raise RuntimeError("❌ cloudflared linki 30 sn içinde alınamadı")

print(f"\n🔗 Queen Editor: {link}\n")
print("⬆️  Linke gir → projeye tıkla → prompt yaz → Üret.\n")
print("📡 Sunucu çalışıyor — BU HÜCREYİ KAPATMA. Canlı log:\n")
try:
    subprocess.run(["tail", "-n", "+1", "-f", FLASK_LOG])
except KeyboardInterrupt:
    print("Hücre durduruldu — Flask hâlâ arka planda (yeni link için tekrar çalıştır).")